# Development workflows on Databricks

**Purpose:** There are multiple ways of writing and running code on Databricks, each with their own advantages. This notebook describes these approaches and gives a rule for when to use each. 

**What this compares.**
1. Interactive workspace: notebooks attached to a cluster, in the browser.
2. Local IDE + Databricks Connect: VS Code and your editor's tooling, remote Spark.
3. Asset Bundle deploy: versioned jobs and pipelines shipped via `databricks bundle`.

## (a) Interactive workspace: explore fast

Open a notebook in the browser, attach a cluster, run cells. Spark, `dbutils`, and the catalog are right there.

**Use it when** you're exploring a dataset, prototyping a model, debugging interactively, or giving a demo. The feedback loop is the shortest you'll get.

The cost of that speed is reproducibility and governance. Cell-by-cell state makes results order-dependent, and the browser is awkward for git, code review, and unit tests. Treat workspace notebooks as a scratchpad, not the source of truth. The moment code matters, move it down the ladder.

## (b) Local IDE + Databricks Connect: engineer properly

The [Databricks VS Code extension](https://docs.databricks.com/dev-tools/vscode-ext/index.html) plus [Databricks Connect](https://docs.databricks.com/dev-tools/databricks-connect/index.html) let you edit code locally, with git, linting, type-checking, a debugger, and `pytest`, while Spark execution happens on a remote cluster. You get real engineering ergonomics without giving up cluster-scale compute.

**Use it when** you're writing code that will be tested, reviewed, and shipped, which is most of serious ML engineering.

**Auth.** Prefer OAuth user-to-machine for local dev; profiles live in `~/.databrickscfg`. The host is your workspace URL (e.g. `https://<your-workspace-host>`):

```bash
databricks auth login --host https://<your-workspace-host>
# creates or updates a named profile; the VS Code extension and CLI both read it
```

Never hardcode tokens (notebook 03 covers secrets). The cell below is a safe connectivity check: it reports status and never hard-fails.

In [ ]:
# Databricks Connect sanity check: guarded so it never hard-fails.
# In a Databricks workspace notebook you already have a `spark` session and don't need this.
# From a local IDE, Databricks Connect builds a session that runs on a remote cluster.
try:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()
    print("Remote Spark reachable:", spark.range(3).count(), "rows computed remotely.")
except Exception as exc:  # ImportError locally, or auth/config errors
    print(f"Databricks Connect not active here ({type(exc).__name__}).")
    print("Expected on GitHub or without a configured profile.")
    print("Set up: databricks auth login, then point the profile at a cluster or serverless.")

## (c) Asset Bundle deploy: ship to production

A [Databricks Asset Bundle](https://docs.databricks.com/dev-tools/bundles/index.html) packages your notebooks, jobs, and pipelines as versioned, declarative config (`databricks.yml`) that deploys identically to dev, staging, and prod. No clicking in the UI; the workspace state becomes a function of code in git.

**Use it when** the work needs to run on a schedule, move across environments, or pass through CI/CD. This is the production destination, covered in depth in notebook 02.

```bash
databricks bundle validate -t dev      # check config against the schema
databricks bundle deploy   -t staging  # create/update jobs + pipelines in staging
databricks bundle run churn_training -t staging
```

*(Shown for shape, not executed here.)*

## The same source flows down the ladder

The point of all three is that one source file can travel the whole way:

1. Sketch it in a workspace notebook.
2. Pull it local with the VS Code extension, refactor logic into `src/`, add `pytest`, and run against remote Spark via Connect.
3. Reference it from a job in `databricks.yml` and deploy the bundle. The same file now runs as a governed, scheduled production job.

| Dimension | Interactive workspace | VS Code + Connect | Asset Bundle deploy |
|---|---|---|---|
| Speed to first result | **Fastest** | Medium | Slowest to set up |
| Reproducibility | Low | Medium | **High** |
| Governance / audit | Low | Medium | **High** |
| Git + code review | Awkward | **Native** | **Native** |
| Unit testing | Hard | **Easy** | **Easy** |
| Best for | Exploration, demos | Engineering, tests | Production, CI/CD |

Rule of thumb: explore in the workspace, build in the IDE, ship with bundles. If a piece of code has outlived the day it was written, it belongs at least one rung down.

## Further reading

[`docs/links.md`](../docs/links.md) carries the full annotated list, one line of *why* per entry. The essentials for this notebook:

- [Databricks extension for VS Code](https://docs.databricks.com/dev-tools/vscode-ext/index.html)
- [Databricks Connect](https://docs.databricks.com/dev-tools/databricks-connect/index.html)
- [Authentication (OAuth / profiles)](https://docs.databricks.com/dev-tools/auth/index.html)
- [Asset Bundles](https://docs.databricks.com/dev-tools/bundles/index.html)